# Extension 02 Compression and tensegrity

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**What happens when force densities are negative ($q < 0$) for bars in compression?**

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** force density $q = f / l$ has the sign of the bar force $f$:
- $q > 0$ pulls nodes together in **tension** (cables).
- $q < 0$ pushes nodes apart in **compression** (vault ribs and tensegrity struts).

In an anchored compression vault, flipping $q$ from positive to negative under gravity load ${\color{#2e9e4a} p_z < 0}$ flips $A = C_N^{T}\,Q\,C_N$ and lifts the hanging net into a funicular arch shell in one linear solve.

In a free standing tensegrity with no fixed supports and ${\color{#2e9e4a} p = 0}$, positive cables and negative struts balance each other inside the nullspace of the stress matrix:

$$ \left(C^{T}\,Q\,C\right) x = 0 $$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_3d.svg"
       style="width:100%; height:auto;"
       alt="Spatial equilibrium of tension and compression bars meeting at a node" />
  <p style="text-align:center;">
    <em>Negative force densities turn hanging cable nets into compression vaults and self stressed tensegrity frames.</em>
  </p>
</div>

**References:**

- **ShellFormFinding**, https://github.com/amandaghassaei/ShellFormFinding
- **Tensegrity Designer**, https://github.com/hermes98761234/tensegrity-designer

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Compression Vault

Take the $6 \times 6$ grid pinned at its four corners at $z = 0\text{ m}$ under downward gravity ${{\color{{#2e9e4a}} p_z = -0.3\text{{ kN}}}}$. Set $q = -2\text{ kN/m}$ inside and $q = -5\text{ kN/m}$ on the four boundary arches. Once solved, Hooke's law $f = EA\,(l - l_0)/l_0$ gives the unstretched timber lath cut lengths:

$$ l_0 = \frac{EA\,l}{EA + f} $$

In [1]:
import numpy as np
from session_py import Color, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, loads green
RED   = Color(0.84, 0.16, 0.16, 1.0)
BLUE  = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)

# nodes and bars of the 6 x 6 grid
n = 7
x, y = np.meshgrid(range(n), range(n))
x = x.flatten().astype(float)
y = y.flatten().astype(float)
z = np.zeros(n * n)

edges = [(k, k+1) for k in range(n*n) if x[k] < n-1] + \
        [(k, k+n) for k in range(n*n) if y[k] < n-1]
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

fixed = [0, n-1, n*(n-1), n*n - 1]
free  = [k for k in range(n*n) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# negative force densities: compression everywhere
q = -2.0 * np.ones(len(edges))
for e, (a, b) in enumerate(edges):
    if (x[a] == x[b] and x[a] in (0, n-1)) or (y[a] == y[b] and y[a] in (0, n-1)):
        q[e] = -5.0
Q = np.diag(q)
p_z = -0.3 * np.ones(len(free))

# equation 6.32
A = C_N.T @ Q @ C_N
x[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ x[fixed])
y[free] = np.linalg.solve(A, 0   - C_N.T @ Q @ C_F @ y[fixed])
z[free] = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z[fixed])

# bar forces f < 0 and unstretched lath cut lengths l_0 > l for EA = 1200 kN
l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)
f = q * l
EA = 1200.0
l_0 = EA * l / (EA + f)

print("vault crown height z24 =", round(z[24], 2), "m")
print("most compressed bar f  =", round(f.min(), 2), "kN")
print("max precut lengthening =", round((l_0 - l).max() * 1000, 2), "mm")

f_abs = np.abs(f)
colors = [Color(0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 1.0) for fv in f_abs]
session = Session("fdm_ext02_vault")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x, y, z]).tolist()), colors, [3.0] * len(edges))
for k in range(n * n):
    pt = session.objects.points[k]
    pt.pointcolor = BLUE if k in fixed else RED
    pt.width = 14.0 if k in fixed else 7.0

show(session, "fdm_ext02_vault")

vault crown height z24 = 0.89 m
most compressed bar f  = -6.5 kN
max precut lengthening = 7.08 mm


# Tensegrity Simplex

A three strut tensegrity prism has 6 free nodes, 9 tension cables ($q > 0$), and 3 compression struts ($q < 0$). For $q_{\text{ring}} = 1$, $q_{\text{vertical}} = \sqrt{3}$, and $q_{\text{strut}} = -\sqrt{3}$, the $6 \times 6$ stress matrix $D = C^{T}\,Q\,C$ has four zero eigenvalues (one shift and three 3D coordinates). Projecting a rough prism onto that nullspace gives an exact self stressed tensegrity.

In [2]:
# 6 nodes (bottom 0,1,2 and top 3,4,5), 9 cables + 3 struts
edges_t = [
    (0, 1), (1, 2), (2, 0),   # bottom cable triangle
    (3, 4), (4, 5), (5, 3),   # top cable triangle
    (0, 3), (1, 4), (2, 5),   # vertical cables
    (0, 4), (1, 5), (2, 3),   # compression struts
]
C_t = np.zeros((12, 6))
for e, (a, b) in enumerate(edges_t):
    C_t[e, a], C_t[e, b] = 1, -1

s3 = np.sqrt(3.0)
q_t = np.array([1, 1, 1, 1, 1, 1, s3, s3, s3, -s3, -s3, -s3], dtype=float)
D = C_t.T @ np.diag(q_t) @ C_t

# 4D nullspace of D = C^T Q C
eigvals, eigvecs = np.linalg.eigh(D)
U_0 = eigvecs[:, np.abs(eigvals) < 1e-10]

# project a rough twisted prism onto the equilibrium nullspace
ang = np.array([0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0])
seed = np.vstack([
    np.column_stack([2.0 * np.cos(ang),       2.0 * np.sin(ang),       np.zeros(3)]),
    np.column_stack([2.0 * np.cos(ang + 2.5), 2.0 * np.sin(ang + 2.5), 2.5 * np.ones(3)]),
])
xyz_t = U_0 @ (U_0.T @ seed)
f_t = q_t * np.linalg.norm(C_t @ xyz_t, axis=1)

print("eigenvalues of D     =", np.round(eigvals, 6))
print("max nodal residual   =", f"{np.abs(D @ xyz_t).max():.2e}", "kN")
print("strut force (comp.)  =", round(f_t[9], 2), "kN")

colors_t = [Color.black()] * 9 + [RED] * 3
widths_t = [2.2] * 9 + [5.5] * 3
session_t = Session("fdm_ext02_tensegrity")
session_t.add_network(Matrix.from_rows(C_t.tolist()), Matrix.from_rows(xyz_t.tolist()), colors_t, widths_t)
for k in range(6):
    pt = session_t.objects.points[k]
    pt.pointcolor = RED
    pt.width = 10.0

show(session_t, "fdm_ext02_tensegrity")

eigenvalues of D     = [-0. -0.  0.  0.  6.  6.]
max nodal residual   = 1.65e-15 kN
strut force (comp.)  = -5.68 kN


# Observation

Negative force densities invert gravity sag into a compression vault, and mixing $q > 0$ cables with $q < 0$ struts produces a floating tensegrity in pure self stress with no supports at all.